# 05 - Validate Results

This notebook validates the expense tracker data pipeline and generates summary statistics.

**Validation checks:**
- Data quality checks
- Record counts and completeness
- Business logic validation
- Analytics verification


## Configuration


In [ ]:
CATALOG_NAME = "main"
SCHEMA_NAME = "expense_tracker"
TABLES = ["categories", "expenses", "budgets"]

print(f"Validating: {CATALOG_NAME}.{SCHEMA_NAME}")


## Check Table Existence


In [ ]:
print("Checking table existence...")
print("="*60)

for table in TABLES:
    full_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table}"
    try:
        df = spark.table(full_name)
        count = df.count()
        print(f"✓ {full_name}: {count} rows")
    except Exception as e:
        print(f"✗ {full_name}: NOT FOUND")
        print(f"  Error: {str(e)}")

print("="*60)


## Data Quality Checks


In [ ]:
-- Check for null values in critical fields
SELECT 
    'categories' as table_name,
    SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) as null_category_id,
    SUM(CASE WHEN category_name IS NULL THEN 1 ELSE 0 END) as null_category_name
FROM main.expense_tracker.categories

UNION ALL

SELECT 
    'expenses' as table_name,
    SUM(CASE WHEN expense_id IS NULL THEN 1 ELSE 0 END) as null_expense_id,
    SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) as null_amount
FROM main.expense_tracker.expenses

UNION ALL

SELECT 
    'budgets' as table_name,
    SUM(CASE WHEN budget_id IS NULL THEN 1 ELSE 0 END) as null_budget_id,
    SUM(CASE WHEN budget_amount IS NULL THEN 1 ELSE 0 END) as null_budget_amount
FROM main.expense_tracker.budgets;


## Referential Integrity Checks


In [ ]:
-- Check for orphaned expenses (expenses with invalid category_id)
SELECT COUNT(*) as orphaned_expenses
FROM main.expense_tracker.expenses e
LEFT JOIN main.expense_tracker.categories c ON e.category_id = c.category_id
WHERE c.category_id IS NULL;


In [ ]:
-- Check for orphaned budgets
SELECT COUNT(*) as orphaned_budgets
FROM main.expense_tracker.budgets b
LEFT JOIN main.expense_tracker.categories c ON b.category_id = c.category_id
WHERE c.category_id IS NULL;


## Business Logic Validation


In [ ]:
-- Check for negative or zero amounts
SELECT 
    COUNT(*) as invalid_amount_count,
    MIN(amount) as min_amount
FROM main.expense_tracker.expenses
WHERE amount <= 0;


In [ ]:
-- Check for future dates
SELECT COUNT(*) as future_date_count
FROM main.expense_tracker.expenses
WHERE expense_date > CURRENT_DATE;


## Summary Statistics


In [ ]:
-- Overall expense statistics
SELECT 
    COUNT(*) as total_expenses,
    COUNT(DISTINCT category_id) as unique_categories,
    MIN(expense_date) as earliest_expense,
    MAX(expense_date) as latest_expense,
    ROUND(SUM(amount), 2) as total_amount,
    ROUND(AVG(amount), 2) as avg_amount,
    ROUND(MIN(amount), 2) as min_amount,
    ROUND(MAX(amount), 2) as max_amount
FROM main.expense_tracker.expenses;


In [ ]:
-- Spending by category
SELECT 
    c.category_name,
    c.category_type,
    COUNT(e.expense_id) as expense_count,
    ROUND(SUM(e.amount), 2) as total_spent,
    ROUND(AVG(e.amount), 2) as avg_expense
FROM main.expense_tracker.expenses e
JOIN main.expense_tracker.categories c ON e.category_id = c.category_id
GROUP BY c.category_name, c.category_type
ORDER BY total_spent DESC;


In [ ]:
-- Budget status check
SELECT 
    c.category_name,
    b.budget_amount,
    COALESCE(SUM(e.amount), 0) as actual_spent,
    ROUND(b.budget_amount - COALESCE(SUM(e.amount), 0), 2) as remaining,
    ROUND((COALESCE(SUM(e.amount), 0) / b.budget_amount * 100), 2) as percent_used,
    CASE 
        WHEN COALESCE(SUM(e.amount), 0) > b.budget_amount THEN 'OVER BUDGET'
        WHEN COALESCE(SUM(e.amount), 0) > b.budget_amount * 0.9 THEN 'WARNING'
        ELSE 'OK'
    END as status
FROM main.expense_tracker.budgets b
JOIN main.expense_tracker.categories c ON b.category_id = c.category_id
LEFT JOIN main.expense_tracker.expenses e ON b.category_id = e.category_id
    AND DATE_TRUNC('month', e.expense_date) = b.month_year
GROUP BY c.category_name, b.budget_amount, b.month_year
ORDER BY percent_used DESC;


## Validation Summary Report


In [ ]:
from datetime import datetime

# Generate validation report
print("="*80)
print("VALIDATION REPORT")
print("="*80)
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Schema: {CATALOG_NAME}.{SCHEMA_NAME}")
print()

# Get table counts
table_stats = {}
for table in TABLES:
    full_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table}"
    try:
        count = spark.table(full_name).count()
        table_stats[table] = count
    except:
        table_stats[table] = 0

print("Table Counts:")
for table, count in table_stats.items():
    print(f"  {table}: {count}")

print()
print("Validation Status:")
all_valid = True

# Check table existence
for table in TABLES:
    if table_stats[table] == 0:
        print(f"  ✗ {table}: No data found")
        all_valid = False
    else:
        print(f"  ✓ {table}: {table_stats[table]} rows")

print()
if all_valid:
    print("✓ All validation checks PASSED")
else:
    print("✗ Some validation checks FAILED")

print("="*80)


## Summary


In [ ]:
print("="*80)
print("VALIDATION COMPLETE")
print("="*80)
print("✓ Data quality checks performed")
print("✓ Referential integrity verified")
print("✓ Business logic validated")
print("✓ Summary statistics generated")
print("\nNext Steps:")
print("  1. Review validation results")
print("  2. Build Databricks App for visualization")
print("  3. Set up automated alerts for budget warnings")
print("="*80)
